In [1]:
%%writefile app.py
import os
import streamlit as st
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import pipeline
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
os.environ["HF_TOKEN"] = "******************"
# ============ Streamlit UI ============
st.title("🎥 YouTube Transcript Q&A (RAG)")

video_url = st.text_input("Enter YouTube Video URL:")
user_question = st.text_area("Enter your question:")

if st.button("Get Answer"):
    try:
        # ---- Extract video_id ----
        if "v=" in video_url:
            video_id = video_url.split("v=")[1].split("&")[0]
        else:
            video_id = video_url.strip()

        # ---- Transcript ----
        yt = YouTubeTranscriptApi()
        transcript_list = yt.fetch(video_id, languages=["en"])
        transcript = " ".join(chunk.text for chunk in transcript_list)

        # ---- Split into chunks ----
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        chunks = splitter.create_documents([transcript])

        # ---- Embeddings + FAISS ----
        embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        vectore_Store = FAISS.from_documents(chunks, embedding_model)
        retriever = vectore_Store.as_retriever(search_type="similarity", search_kwargs={"k": 1})

        # ---- HuggingFace Model ----
        # 🔑 Set Hugging Face Token


        generator = pipeline(
            "text2text-generation",
            model="google/flan-t5-large",   # base model is open, no token required
            max_new_tokens=300,
            temperature=0.3,
             token=os.environ["HF_TOKEN"]
        )
        llm = HuggingFacePipeline(pipeline=generator)

        # ---- Prompt ----
        prompt = PromptTemplate(
            template="""
            You are a helpful assistant.
            Answer ONLY from the provided transcript context.
            Explain your answer in detail, step by step.
            Do not answer just yes/no.

            {context}
            Question: {question}
            """,
            input_variables=['context', 'question']
        )

        def format_docs(retrieved_docs):
            return "\n\n".join(doc.page_content for doc in retrieved_docs)

        # ---- Chain ----
        parallel_chain = RunnableParallel({
            'context': retriever | RunnableLambda(format_docs),
            'question': RunnablePassthrough()
        })

        parser = StrOutputParser()
        main_chain = parallel_chain | prompt | llm | parser

        # ---- Answer ----
        if user_question:
            answer = main_chain.invoke(user_question)
            st.subheader("📖 Answer")
            st.write(answer)
        else:
            st.warning("⚠️ Please enter a question.")

    except TranscriptsDisabled:
        st.error("❌ No captions available for this video.")
    except Exception as e:
        st.error(f"⚠️ Error: {str(e)}")


Overwriting app.py


In [2]:
pip install streamlit


In [3]:
!pip install youtube-transcript-api


In [4]:
pip install streamlit youtube-transcript-api langchain langchain-community langchain-huggingface faiss-cpu transformers tiktoken


Note: you may need to restart the kernel to use updated packages.


In [5]:
!pip install sentence-transformers


In [6]:
!pip install tf-keras


In [7]:
!streamlit run app.py 


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.100.7:8501



2025-09-17 11:22:47.663791: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-17 11:22:56.978686: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

2025-09-17 11:23:19.719 Examining the path of torch.classes raised: Tried to instantiate class '__path__._path', but it does not exist! Ensure that it is registered via torch::class_


2025-09-17 11:24:51.693 Examining the path of torch.classes raised: Tried to instantiate class '__path__._path', but it does not exist! Ensure that it is registered via torch::class_
Xet Storage is enabled for this repo, but t